<a href="https://colab.research.google.com/github/lcontrerasroa/ipatracer/blob/main/ipatrace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U fonttools

import os, unicodedata, zipfile, urllib.request
from fontTools.ttLib import TTFont
from fontTools.pens.svgPathPen import SVGPathPen
from fontTools.pens.boundsPen import BoundsPen

In [ ]:
SYMBOLS_BASE = """\
a
æ
ɑ
ɒ
ɔ
e
ə
ɜ
ʌ
i
ɪ
u
ʊ
p
b
t
d
k
g
f
v
θ
ð
s
z
ʃ
ʒ
h
m
n
ŋ
l
ɫ
r
ɹ
j
w
ɾ
ʔ
ˈ
ˌ
ː
"""

def write_list(path, txt):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(txt)

LIST_PATH = "/content/symbols_base.txt"
write_list(LIST_PATH, SYMBOLS_BASE)
print("Liste écrite:", LIST_PATH)


Liste écrite: /content/symbols_base.txt


In [ ]:
def get_andika_ttf():
    os.makedirs("/content/fonts", exist_ok=True)
    url = "https://software.sil.org/downloads/r/andika/Andika-6.200.zip"
    zip_path = "/content/fonts/andika.zip"
    print("Téléchargement:", url)
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall("/content/fonts")
    for root, _, files in os.walk("/content/fonts"):
        for f in files:
            if f.lower().endswith(".ttf") and "Andika" in f:
                return os.path.join(root, f)
    return None

FONT_PATH = get_andika_ttf()
assert FONT_PATH and os.path.exists(FONT_PATH), "Andika introuvable. Télécharge manuellement et réessaie."
print("FONT_PATH =", FONT_PATH)


Téléchargement: https://software.sil.org/downloads/r/andika/Andika-6.200.zip
FONT_PATH = /content/fonts/Andika-6.200/Andika-Bold.ttf


In [ ]:
def build_cmap(tt: TTFont):
    cmap = {}
    try:
        cmap.update(tt.getBestCmap())  # cp -> glyphName
    except Exception:
        pass
    for table in tt["cmap"].tables:
        if table.isUnicode():
            for cp, val in table.cmap.items():
                if isinstance(val, int):
                    try: val = tt.getGlyphName(val)
                    except Exception: continue
                cmap[cp] = val
    return cmap

def char_to_glyph(tt: TTFont, ch: str, cmap: dict):
    cp = ord(ch)
    if cp in cmap:
        return cmap[cp]
    decomp = unicodedata.normalize("NFD", ch)
    if decomp:
        cp0 = ord(decomp[0])
        if cp0 in cmap: return cmap[cp0]
    return None

def glyph_path_d(tt: TTFont, glyph_name: str):
    gs = tt.getGlyphSet()
    if glyph_name not in gs: return None
    pen = SVGPathPen(gs)
    gs[glyph_name].draw(pen)
    return pen.getCommands()

def glyph_bounds(tt: TTFont, glyph_name: str):
    gs = tt.getGlyphSet()
    pen = BoundsPen(gs)
    gs[glyph_name].draw(pen)
    return pen.bounds or (0,0,0,0)  # (xmin, ymin, xmax, ymax)

def safe_filename(ch: str):
    return ch if (ch and ch.isprintable() and ch not in r'\/:*?"<>| ') else f"u{ord(ch):04X}"

def write_svg_uniform(d_path: str, tt: TTFont, glyph_name: str, out_path: str,
                      viewbox_em: int = 1000, stroke_px: int = 12, padding_pct: float = 6.0):
    xmin, ymin, xmax, ymax = glyph_bounds(tt, glyph_name)
    gw = max(1, xmax - xmin)
    gh = max(1, ymax - ymin)

    pad = viewbox_em * (padding_pct / 100.0)
    vb_w = viewbox_em + 2*pad
    vb_h = viewbox_em + 2*pad

    scale = viewbox_em / max(gw, gh)  # échelle UNIFORME
    # centrer et basculer l'axe Y (SVG)
    trans_x = pad + (viewbox_em - gw*scale)/2 - xmin*scale
    trans_y = pad + (viewbox_em - gh*scale)/2 + ymax*scale

    svg = f'''<?xml version="1.0" encoding="UTF-8"?>
<svg width="{int(vb_w)}" height="{int(vb_h)}" viewBox="0 0 {vb_w} {vb_h}" xmlns="http://www.w3.org/2000/svg">
  <rect x="0" y="0" width="{vb_w}" height="{vb_h}" fill="white"/>
  <g transform="translate({trans_x:.3f},{trans_y:.3f}) scale({scale:.5f},{-scale:.5f})">
    <path d="{d_path}" fill="none" stroke="#000" stroke-width="{stroke_px}" stroke-linecap="round" stroke-linejoin="round"/>
  </g>
</svg>'''
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(svg)

def export_list_to_svgs(font_path: str, list_path: str, outdir: str,
                        stroke_px: int = 12, padding_pct: float = 6.0, size_em: int = 1000):
    tt = TTFont(font_path)
    cmap = build_cmap(tt)
    with open(list_path, "r", encoding="utf-8") as f:
        chars = [ln.strip() for ln in f if ln.strip()]
    ok = miss = 0
    for ch in chars:
        gname = char_to_glyph(tt, ch, cmap)
        if not gname:
            print(f"[!] Manquant: {repr(ch)} U+{ord(ch):04X}"); miss += 1; continue
        d = glyph_path_d(tt, gname)
        if not d:
            print(f"[!] Vide: {repr(ch)} -> {gname}"); miss += 1; continue
        out_path = os.path.join(outdir, f"{safe_filename(ch)}.svg")
        write_svg_uniform(d, tt, gname, out_path, viewbox_em=size_em, stroke_px=stroke_px, padding_pct=padding_pct)
        ok += 1; print(f"[OK] {ch} -> {out_path}")
    print(f"\nTerminé: {ok} SVG générés, {miss} manquants.")


In [ ]:
OUTDIR = "/content/svgs_andika_base"
export_list_to_svgs(FONT_PATH, LIST_PATH, outdir=OUTDIR, stroke_px=12, padding_pct=6.0, size_em=1000)

# Petit récap
print("Sortie:", OUTDIR, "— fichiers:", len([f for f in os.listdir(OUTDIR) if f.endswith(".svg")]))


[OK] a -> /content/svgs_andika_base/a.svg
[OK] æ -> /content/svgs_andika_base/æ.svg
[OK] ɑ -> /content/svgs_andika_base/ɑ.svg
[OK] ɒ -> /content/svgs_andika_base/ɒ.svg
[OK] ɔ -> /content/svgs_andika_base/ɔ.svg
[OK] e -> /content/svgs_andika_base/e.svg
[OK] ə -> /content/svgs_andika_base/ə.svg
[OK] ɜ -> /content/svgs_andika_base/ɜ.svg
[OK] ʌ -> /content/svgs_andika_base/ʌ.svg
[OK] i -> /content/svgs_andika_base/i.svg
[OK] ɪ -> /content/svgs_andika_base/ɪ.svg
[OK] u -> /content/svgs_andika_base/u.svg
[OK] ʊ -> /content/svgs_andika_base/ʊ.svg
[OK] p -> /content/svgs_andika_base/p.svg
[OK] b -> /content/svgs_andika_base/b.svg
[OK] t -> /content/svgs_andika_base/t.svg
[OK] d -> /content/svgs_andika_base/d.svg
[OK] k -> /content/svgs_andika_base/k.svg
[OK] g -> /content/svgs_andika_base/g.svg
[OK] f -> /content/svgs_andika_base/f.svg
[OK] v -> /content/svgs_andika_base/v.svg
[OK] θ -> /content/svgs_andika_base/θ.svg
[OK] ð -> /content/svgs_andika_base/ð.svg
[OK] s -> /content/svgs_andika_bas

In [ ]:
ZIP_PATH = "/content/ipa_andika_svgs.zip"
!zip -r "$ZIP_PATH" "$OUTDIR" -q
from google.colab import files
files.download(ZIP_PATH)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>